# GraphGrasp — Cross-Embodiment Retargeting Training

**Architecture:** E_h (GAT) + E_X + D_X (MLP) + E_r + D_r (linear) + ShadowFK  
**Losses:** L_contrastive + L_rec + L_ltc + L_temporal (Yan & Lee 2026)  
**Dataset:** hograspnet_retarget.csv (Dong quats + raw XYZ, ~1.5M frames)  

**Setup checklist:**
- Runtime: T4 GPU
- Drive mounted with `hograspnet_retarget.csv` in `AIST-hand/data/processed/`
- Drive mounted with `shadow_hand_right.urdf` in `AIST-hand/dex-urdf/robots/hands/shadow_hand/`

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import os

DRIVE_DATA_DIR   = '/content/drive/MyDrive/AIST-hand/data'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
GITHUB_TOKEN     = ''  # paste your token here
GITHUB_USER      = 'isapedraza'
REPO_NAME        = 'AIST-hand'

# Training hyperparameters
EPOCHS     = 50
BATCH_SIZE = 256
LR         = 1e-3
Z_DIM      = 32
SHARED_DIM = 256
OMEGA      = 1.0   # D_ee weight in similarity metric
ALPHA      = 0.05  # triplet margin
SAVE_EVERY = 5
EVAL_EVERY = 5     # compute RS/NDS/NVS on test set every N epochs
TEST_FRAC  = 0.2   # 80/20 sequence-level split

RUN_NAME = 'retarget_v1'

RESUME_FROM = None  # e.g. 'epoch010.pt' to resume; None = train from scratch

print(f'Run: {RUN_NAME}')
print(f'Data dir: {DRIVE_DATA_DIR}')


## 3. Clone repository

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

## 4. Install dependencies

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

!pip install -q torch-geometric
!pip install -q pytorch-kinematics
!pip install -q pandas numpy
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Link data from Drive

In [ ]:
processed_dir = f'{REPO_DIR}/grasp-model/data/processed'
os.makedirs(processed_dir, exist_ok=True)

# Symlink retarget CSV (2 GB — avoid copying)
retarget_csv = f'{processed_dir}/hograspnet_retarget.csv'
if not os.path.exists(retarget_csv):
    src = f'{DRIVE_DATA_DIR}/processed/hograspnet_retarget.csv'
    os.symlink(src, retarget_csv)
    print(f'Linked: {retarget_csv}')
else:
    print('retarget CSV already linked.')

# Symlink dex-urdf (for ShadowFK)
urdf_dir = f'{REPO_DIR}/dex-urdf'
if not os.path.exists(urdf_dir):
    src = f'{DRIVE_DATA_DIR}/../dex-urdf'
    if os.path.exists(src):
        os.symlink(src, urdf_dir)
        print(f'Linked: {urdf_dir}')
    else:
        print(f'WARNING: dex-urdf not found at {src}. Set URDF_PATH manually below.')
else:
    print('dex-urdf already linked.')

# Override URDF path if needed
URDF_PATH = None  # set to explicit path if symlink failed, e.g. '/content/drive/MyDrive/shadow_hand_right.urdf'

## 6. Train

In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/grasp-model/src')

import torch
import torch.optim as optim
from pathlib import Path
from torch_geometric.loader import DataLoader

from grasp_gcn.network.human_encoder import HumanEncoder
from grasp_gcn.network.cross_embodiment import CrossEmbodimentEncoder, CrossEmbodimentDecoder
from grasp_gcn.network.robot_embedding import make_shadow_hand_embedding
from grasp_gcn.network.shadow_fk import ShadowFK
from grasp_gcn.network.losses import loss_ltc, loss_rec, loss_contrastive, loss_temporal, HUMAN_QUAT_IDX, ROBOT_QUAT_IDX
from grasp_gcn.dataset.retarget_dataset import RetargetDataset

LAMBDA_C    = 10.0
LAMBDA_REC  = 5.0
LAMBDA_LTC  = 1.0
LAMBDA_TEMP = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def normalize_human_tips(xyz):
    wrist    = xyz[:, 0, :]
    tips     = xyz[:, [4, 8, 12, 16, 20], :]
    mcp_mid  = xyz[:, 9, :]
    hand_len = (mcp_mid - wrist).norm(dim=-1, keepdim=True).clamp(min=1e-6)
    return (tips - wrist.unsqueeze(1)) / hand_len.unsqueeze(1)

# Models
E_h = HumanEncoder(in_dim=4, hidden_dim=32, heads=4, z_dim=Z_DIM).to(device)
E_X = CrossEmbodimentEncoder(shared_dim=SHARED_DIM, z_dim=Z_DIM).to(device)
D_X = CrossEmbodimentDecoder(z_dim=Z_DIM, shared_dim=SHARED_DIM).to(device)
E_r = make_shadow_hand_embedding(shared_dim=SHARED_DIM).to(device)

params = list(E_h.parameters()) + list(E_X.parameters()) + \
         list(D_X.parameters()) + list(E_r.parameters())
optimizer = optim.Adam(params, lr=LR)

fk = ShadowFK(urdf_path=URDF_PATH, device=device)

# Resume from checkpoint
start_epoch = 1
if RESUME_FROM:
    ckpt_path = Path(DRIVE_OUTPUT_DIR) / RUN_NAME / RESUME_FROM
    ckpt_resume = torch.load(ckpt_path, map_location=device)
    E_h.load_state_dict(ckpt_resume['E_h'])
    E_X.load_state_dict(ckpt_resume['E_X'])
    D_X.load_state_dict(ckpt_resume['D_X'])
    E_r.load_state_dict(ckpt_resume['E_r'])
    optimizer.load_state_dict(ckpt_resume['optimizer'])
    start_epoch = ckpt_resume['epoch'] + 1
    print(f'Resumed from {RESUME_FROM}, starting at epoch {start_epoch}')

# 80/20 sequence-level train/test split
CSV_PATH = f'{REPO_DIR}/grasp-model/data/processed/hograspnet_retarget.csv'
train_ds, test_ds = RetargetDataset.train_test_split(CSV_PATH, test_frac=TEST_FRAC)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds):,} pairs | {len(train_loader)} batches/epoch')
print(f'Test:  {len(test_ds):,} pairs  | {len(test_loader)} batches')

# Output dir
ckpt_dir = Path(DRIVE_OUTPUT_DIR) / RUN_NAME
ckpt_dir.mkdir(parents=True, exist_ok=True)


@torch.no_grad()
def evaluate():
    """Compute RS, NDS, NVS on test set (Yan & Lee 2026 metrics).

    RS  = avg D_R(human, retargeted robot)  -- joint rotation alignment
    NDS = avg D_ee(human, retargeted robot) -- normalized fingertip distance
    NVS = avg ||v_human - v_robot_ee||_2    -- velocity consistency
    Lower is better for all three.
    """
    E_h.eval(); E_X.eval(); D_X.eval(); E_r.eval()
    rs_sum = nds_sum = nvs_sum = 0.0
    n = 0

    for batch in test_loader:
        graph_t  = batch['graph_t'].to(device)
        graph_t1 = batch['graph_t1'].to(device)
        xyz_t    = batch['xyz_t'].to(device)
        xyz_t1   = batch['xyz_t1'].to(device)
        quats_t  = batch['quats_t'].to(device)   # [B, 20, 4]
        B = xyz_t.shape[0]

        # Retarget human frame t -> robot qpos
        z_h_t  = E_h(graph_t.x,  graph_t.edge_index,  graph_t.batch)
        z_h_t1 = E_h(graph_t1.x, graph_t1.edge_index, graph_t1.batch)
        qpos_t  = E_r.decode(D_X(z_h_t))
        qpos_t1 = E_r.decode(D_X(z_h_t1))

        # FK on retargeted poses
        r_tips_t,  r_quats_t  = fk.forward(qpos_t)
        r_tips_t1, _          = fk.forward(qpos_t1)

        # RS: D_R between 20 comparable joint pairs (Eq. 3)
        hq = quats_t[:, HUMAN_QUAT_IDX, :]           # [B, 20, 4]
        rq = r_quats_t[:, ROBOT_QUAT_IDX, :]          # [B, 20, 4]
        dot = (hq * rq).sum(-1)                        # [B, 20]
        rs_batch = (1 - dot ** 2).sum(-1).mean().item()

        # NDS: D_ee between normalized fingertips (Eq. 4)
        h_tips = normalize_human_tips(xyz_t)[:, 1, :]  # mftip [B, 3]
        r_tips = fk.normalize_tips(r_tips_t)[:, 1, :]  # mftip [B, 3]
        nds_batch = (h_tips - r_tips).norm(dim=-1).mean().item()

        # NVS: velocity difference (Eq. 9 / L_temporal)
        hand_len = (xyz_t[:, 9, :] - xyz_t[:, 0, :]).norm(dim=-1, keepdim=True).clamp(1e-6)
        v_human  = ((xyz_t1[:, 12, :] - xyz_t[:, 12, :]) -
                    (xyz_t1[:, 0,  :] - xyz_t[:, 0,  :])) / hand_len
        v_robot  = fk.normalize_tips(r_tips_t1)[:, 1, :] - fk.normalize_tips(r_tips_t)[:, 1, :]
        nvs_batch = (v_human - v_robot).norm(dim=-1).mean().item()

        rs_sum  += rs_batch
        nds_sum += nds_batch
        nvs_sum += nvs_batch
        n += 1

    E_h.train(); E_X.train(); D_X.train(); E_r.train()
    return rs_sum / n, nds_sum / n, nvs_sum / n


# Training loop
for epoch in range(start_epoch, EPOCHS + 1):
    total = total_lc = total_rec = total_ltc = total_temp = 0.0
    n = 0

    for batch in train_loader:
        graph_t  = batch['graph_t'].to(device)
        graph_t1 = batch['graph_t1'].to(device)
        xyz_t    = batch['xyz_t'].to(device)
        xyz_t1   = batch['xyz_t1'].to(device)
        quats_t  = batch['quats_t'].to(device)
        B = xyz_t.shape[0]

        # Human frame t
        z_h      = E_h(graph_t.x, graph_t.edge_index, graph_t.batch)
        shared_h = D_X(z_h)
        z_rt     = E_X(shared_h)
        l_ltc    = loss_ltc(z_h, z_rt)

        # Robot random sample
        qpos              = fk.sample(B)
        r_tips_raw, r_quats = fk.forward(qpos)
        r_tips            = fk.normalize_tips(r_tips_raw)
        z_r               = E_X(E_r.encode(qpos))
        qpos_hat          = E_r.decode(D_X(z_r))
        l_rec             = loss_rec(qpos, qpos_hat)

        # Contrastive
        z_all  = torch.cat([z_h, z_r], dim=0)
        h_tips = normalize_human_tips(xyz_t)
        l_c    = loss_contrastive(z_all, quats_t, r_quats, h_tips, r_tips,
                                  alpha=ALPHA, omega=OMEGA)

        # Temporal
        z_h1         = E_h(graph_t1.x, graph_t1.edge_index, graph_t1.batch)
        qpos_t_ret   = E_r.decode(D_X(z_h))
        qpos_t1_ret  = E_r.decode(D_X(z_h1))
        tips_t,  _   = fk.forward(qpos_t_ret.detach())
        tips_t1, _   = fk.forward(qpos_t1_ret.detach())
        ee_t         = fk.normalize_tips(tips_t)[:, 1, :]
        ee_t1        = fk.normalize_tips(tips_t1)[:, 1, :]
        l_temp       = loss_temporal(xyz_t, xyz_t1, ee_t, ee_t1)

        loss = LAMBDA_C*l_c + LAMBDA_REC*l_rec + LAMBDA_LTC*l_ltc + LAMBDA_TEMP*l_temp

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total      += loss.item()
        total_lc   += l_c.item()
        total_rec  += l_rec.item()
        total_ltc  += l_ltc.item()
        total_temp += l_temp.item()
        n += 1

    a = lambda x: x / n
    log = (f'Epoch {epoch:03d} | loss={a(total):.4f} '
           f'Lc={a(total_lc):.4f} Lrec={a(total_rec):.4f} '
           f'Lltc={a(total_ltc):.4f} Ltemp={a(total_temp):.4f}')

    if epoch % EVAL_EVERY == 0:
        rs, nds, nvs = evaluate()
        log += f' | RS={rs:.4f} NDS={nds:.4f} NVS={nvs:.4f}'

    print(log)

    if epoch % SAVE_EVERY == 0:
        ckpt = ckpt_dir / f'epoch{epoch:03d}.pt'
        torch.save({
            'epoch': epoch,
            'E_h':  E_h.state_dict(),
            'E_X':  E_X.state_dict(),
            'D_X':  D_X.state_dict(),
            'E_r':  E_r.state_dict(),
            'optimizer': optimizer.state_dict(),
        }, ckpt)
        print(f'  Saved {ckpt}')

print('Training complete.')
